# Taller: Detección de Objetos con YOLO

### Sebastián Palma

## 1. Velocidad vs. Precisión

### Comparar los tiempos de entrenamiento e inferencia entre las variantes yolo11s (small) y yolo11n (nano).

Al entrenar ambos modelos con el mismo dataset, los tiempos de entrenamiento fueron similares: 8.3 minutos para small y 9.3 para nano. Esto se explica porque en datasets pequeños el cuello de botella está en la carga de datos más que en el modelo en sí. En inferencia la diferencia fue más evidente, con small procesando cada imagen en 3.4 ms frente a los 6.8 ms de nano. En precisión, small alcanzó un mAP50 de 0.850 y mAP50-95 de 0.652, mientras que nano al tener menos de un tercio de los parámetros suele quedar entre un 8% y 12% por debajo, lo que le cuesta a la hora de distinguir correctamente las 20 clases del dataset.

### ¿En qué escenarios es más conveniente usar cada variante (nano, small, medium, large, extra-large)? Cantidad de datos, recursos computacionales, etc.

La elección de la variante depende principalmente de los recursos disponibles y el nivel de precisión requerido. Nano es ideal para dispositivos con recursos limitados como Raspberry Pi o celulares sin GPU, y también sirve para prototipar rápidamente antes de escalar. Small ofrece el mejor equilibrio entre velocidad y precisión, siendo la opción más versátil para laptops con GPU o servidores modestos en producción. Medium cobra sentido cuando se tienen datasets más grandes y una GPU más potente que permita al modelo aprovechar mejor los datos. Large y Extra Large quedan reservadas para aplicaciones críticas donde la precisión es lo primero, como diagnóstico médico o sistemas avanzados de seguridad, y donde la infraestructura de cómputo no es una limitante.

## 2. mAP50 vs. mAP50–95

### ¿Qué significa que mAP50 evalúe solo intersecciones sobre uniones (IoU) mayores o iguales a 0.5?

Que mAP50 use un umbral de IoU de 0.5 significa que una detección se considera correcta cuando la caja predicha y la real se superponen al menos en un 50%, lo que le da al modelo cierto margen de error en la localización. En la práctica esto evalúa si el modelo identificó correctamente el objeto y lo ubicó de forma aproximada, sin exigir precisión exacta en los bordes. En el experimento clases como Rat y Rhino alcanzaron valores cercanos a 1 por ser visualmente distintivas, mientras que Panda y Camel obtuvieron resultados más bajos probablemente por la variabilidad de poses o similitud con otras clases.

### ¿Qué información adicional proporciona evaluar el rango mAP50–95 al medir el rendimiento del modelo?

A diferencia de mAP50, la métrica mAP50-95 es más estricta porque promedia el rendimiento en umbrales de IoU desde 0.50 hasta 0.95, exigiendo que la caja predicha coincida con mucha mayor precisión con la real. En el experimento el modelo obtuvo un mAP50-95 de 0.652 frente al mAP50 de 0.850, una brecha que indica que el modelo reconoce bien los animales y los localiza de forma aproximada, pero las cajas no siempre se ajustan con precisión a sus bordes reales.

## 3. Hiperparámetros críticos

### ¿Cómo afectan los valores de batch (tamaño de lote), imgsz (tamaño de imagen) y patience (paciencia para detener el entrenamiento) al tiempo de entrenamiento, uso de memoria y convergencia del modelo?

El batch de 16 generó 85 pasos por época con tiempos de entre 18 y 25 segundos y un consumo de 4.09 GB de VRAM. Batches más grandes aceleran el entrenamiento pero aumentan el uso de memoria y tienden a producir gradientes más estables, mientras que batches pequeños introducen más ruido aunque en algunos casos ayudan a evitar mínimos locales. El imgsz de 640 ofreció un buen equilibrio entre velocidad y detalle, ya que reducirlo agilizaría el proceso pero perdería precisión en objetos pequeños, y aumentarlo mejoraría ese detalle a costa de más memoria y un batch más pequeño. El patience de 5 permitió que las 20 épocas se completaran sin interrupciones, lo cual fue clave cuando el mAP50 bajó temporalmente entre las épocas 3 y 4 antes de recuperarse y llegar a 0.850.

## 4. Data Augmentation

### ¿Qué técnicas de aumento de datos podrían mejorar la capacidad del modelo para generalizar en diferentes escenarios?

Varias técnicas podrían ayudar al modelo a generalizar mejor en escenarios que el dataset no cubre completamente. El random crop agresivo obligaría al modelo a detectar animales visibles solo parcialmente, algo frecuente en fauna silvestre donde el animal aparece cortado por el encuadre o cubierto por vegetación. Mixup combinaría dos imágenes en una sola, ayudando a diferenciar clases visualmente similares como Panda y Camel, que fueron de las más confundidas durante el entrenamiento. La rotación y cambios de perspectiva generarían ángulos poco comunes como tomas de drones o cámaras trampa, haciendo al modelo más robusto ante escenas distintas a las frontales y laterales del dataset. Por último, el ruido gaussiano y la variación extrema de escala simularían condiciones de baja calidad de imagen y animales a distancias muy distintas, mejorando el desempeño en escenarios reales con hardware menos especializado.